In [ ]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch
from resnet20 import *
from PIL import Image

In [78]:
val_transforms = transforms.Compose([
    transforms.Resize((32,32)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_set = ImageFolder(r"C:\github\dataset\archive\test", transform=val_transforms)

test_loader = DataLoader(
    test_set,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
model =resnet20(2)
model.load_state_dict(torch.load('model/best_model_rn20.pth', map_location=device))
model.eval()
model = model.to(device)  


C:\Users\nsyn1\AppData\Local\Temp\ipykernel_35372\1692405756.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('model/best_model_rn20.pth'

In [80]:
image = Image.open(r'C:\github\DatathonxAI\backend\images\FAKE5.jpg')
image = image.resize((1024,1024))
image_tensor = transforms.ToTensor()(image)

In [ ]:
H,W = image_tensor.shape[1],image_tensor.shape[2]
STRIDE = 16
PATCH_SIZE = 32

patches =[]
for i in range(0,H-PATCH_SIZE+1,STRIDE):
    for j in range(0,W-PATCH_SIZE+1,STRIDE):
        patch = image_tensor[:,i:i+PATCH_SIZE,j:j+STRIDE]
        patches.append(patch)

patches = torch.stack(patches).to('cuda')

In [ ]:
predictions = model(patches)
probabilities = torch.softmax(predictions, dim=1)

In [85]:
confidence, predicted_labels = torch.max(probabilities, dim=1)  # max confidence and class


In [86]:
fake_indexes = torch.where(predicted_labels == 1)
real_indexes = torch.where(predicted_labels == 0)

In [87]:
fake_conf = confidence[fake_indexes].mean()
real_conf = confidence[real_indexes].mean()
total_conf = fake_conf + real_conf

In [88]:
fake_weight= fake_conf/total_conf
real_weight = real_conf/total_conf

In [89]:
fake_amount = len(torch.where(predicted_labels == 1)[0])
real_amount = len(torch.where(predicted_labels == 0)[0])

In [90]:
authenticity_score = (real_amount * real_weight) - (fake_amount * fake_weight)
final_score = authenticity_score/len(predicted_labels)

In [91]:
if final_score < -0.1:
    print("Prediction: Fake")
    print(f"Confidence: {abs(final_score)*100:.2f}%")
elif final_score > 0.1:
    print("Prediction: Real")
    print(f"Confidence: {final_score*100:.2f}%")
else:
    print("Prediction: Mixed")


Prediction: Real
Confidence: 12.07%
